In [ ]:
%load_ext autoreload
%autoreload 2


import numpy as np
import pandas as pd
import torch
import pydicom
import matplotlib.pyplot as plt

from pathlib import Path

# MONAI imports
import monai
from monai.data import Dataset, CacheDataset, DataLoader, PILReader
from monai.transforms import (
    LoadImage, LoadImaged, Resized, Compose, SaveImage, 
    Spacingd, SpatialCropd, ResizeWithPadOrCropd
)

import numpy as np
from monai.transforms import (
    Compose,
    LoadImaged,
    Transposed,
    NormalizeIntensityd,
    MapTransform,
    ScaleIntensityRangePercentilesd,
    RandAffined, RandGaussianNoised, 
    RandStdShiftIntensityd, RandScaleIntensityd, RandAdjustContrastd, RandHistogramShiftd,
    ScaleIntensityd, Lambdad,
    LoadImage, Transpose
)

from torch.utils.data import DataLoader
from tqdm.notebook import tqdm



import landmarker
import landmarker.datasets
from landmarker.datasets import get_cepha_landmark_datasets
from landmarker.heatmap import GaussianHeatmapGenerator
from landmarker.models import OriginalSpatialConfigurationNet
from landmarker.losses import GaussianHeatmapL2Loss
from torch.utils.data import DataLoader
from landmarker.visualize import inspection_plot


from landmarker.data import LandmarkDataset

#   My stuff
import ra_utils
import ra_utils.data.data_utils
from  ra_utils.data.data_utils import (
    extract_extras_from_filename, 
    extract_extras_from_abspath
)

import ra_utils.visualization.plot_landmarks
import ra_utils.data
import ra_utils.data.data_handler
import ra_utils.data.dataloader_CR_landmarks
import ra_utils.visualization.plot_landmarks #.plot_landmarks
import pydicom
import numpy as np

import pydicom
import numpy as np
import pandas as pd

def get_dicom_info(dicom_paths):
    """
    Given a list of DICOM file paths, returns a DataFrame where each row 
    corresponds to one file, indexed by the file path. 
    The columns include:
      - dim_original              : Shape of the pixel array
      - pixel_spacing            : Pixel spacing if present in the DICOM metadata
      - intensity_range_original : (min, max) of the pixel data
      - intensity_005            : 0.05% intensity quantile
      - intensity_500            : 50% intensity quantile (median)
      - intensity_995            : 99.5% intensity quantile
    """
    records = []

    for path in dicom_paths:
        try:
            ds = pydicom.dcmread(path)
            pixel_array = ds.pixel_array

            dim_original = pixel_array.shape
            pixel_spacing = getattr(ds, "PixelSpacing", None)
            
            # Basic intensity range
            intensity_range_original = (pixel_array.min(), pixel_array.max())
            
            # Compute quantiles: 0.05%, 50%, and 99.5%
            # Note: 0.05% = 0.0005 in decimal, 
            #       50%   = 0.5,
            #       99.5% = 0.995
            q_005, q_500, q_995 = np.quantile(pixel_array, [0.0005, 0.5, 0.995])
            
            records.append({
                "file_path": path,
                "dim_original": dim_original,
                "pixel_spacing": pixel_spacing,
                "intensity_min": intensity_range_original[0],
                "intensity_005": q_005,
                "intensity_500": q_500,
                "intensity_995": q_995,
                "intensity_max": intensity_range_original[1]
            })
        except Exception as e:
            # In case a file is not a valid DICOM or any other read error occurs
            print(f"Warning: Could not process {path}. Error: {e}")
            continue

    # Convert to DataFrame and set the index to the file path
    df = pd.DataFrame(records).set_index("file_path")
    return df


import numpy as np
from sklearn.model_selection import KFold

import numpy as np
from sklearn.model_selection import GroupKFold

def generate_split_dictionary(
    df_splits_input,
    cw_splits=5, 
    train_val_test_split_proportions=(0.6, 0.2, 0.2),
    random_seed=42
):
    """
    Returns a dictionary with two major parts:
      1) 'cv': a list of cross-validation folds, where each element is a dict
         containing 'train' and 'test' keys with lists of filenames.
         - Each fold is based on patients (i.e., GroupKFold).
      2) The final train/test1/test2 split is also performed on the patient level.

    Parameters
    ----------
    df_splits_input : pd.DataFrame
        A DataFrame containing (at least) two columns:
          - "filename" whose values serve as unique IDs for images
          - "patient_id" which groups images by patient
    cw_splits : int, default=5
        Number of cross-validation folds to generate.
    train_val_test_split_proportions : tuple of floats, default=(0.6, 0.2, 0.2)
        Proportions used for final dataset splitting.
    random_seed : int, default=42
        Random seed for reproducibility.

    Returns
    -------
    dict
        A dictionary with the structure:
            {
                "cv": [
                    {"train": [...], "test": [...]},
                    {"train": [...], "test": [...]},
                    ...
                ],
                "training": [...],
                "test1": [...],
                "test2": [...]
            }

        where "cv" is a list of cross-validation folds
        and "training"/"test1"/"test2" are the final splits.
    """

    # Ensure the proportions sum to 1.0
    assert abs(sum(train_val_test_split_proportions) - 1.0) < 1e-9, \
        "train_val_test_split_proportions must sum to 1.0"

    # -------------------------------------------------
    # 1) Unique patients (group identifiers)
    # -------------------------------------------------
    # We'll split on patient_id, not on every filename
    unique_patients = df_splits_input["patient_id"].unique()
    n_patients = len(unique_patients)

    # Shuffle the patient IDs
    rng = np.random.RandomState(random_seed)
    shuffled_patients = rng.permutation(unique_patients)

    # -------------------------------------------------
    # 2) Cross-validation with GroupKFold
    # -------------------------------------------------
    # Prepare a GroupKFold to ensure that all samples from each patient end up
    # in the same fold
    gkf = GroupKFold(n_splits=cw_splits)

    cv_splits = []
    # We need to pass 'X' (features) and 'y' (labels) to GroupKFold, but we only 
    # use them for indexing. The important part is the 'groups' argument.
    # For convenience, let's create an array from the shuffled patient list 
    # so we can do indexing easily.
    # 
    # Trick: We'll just index from 0...n_patients for the patients in "shuffled_patients"
    # and then map each index back to a patient ID and then to the actual filenames.
    # 
    # We need a dictionary from patient_id -> index in the 'shuffled_patients' array.
    patient_to_idx = {p: i for i, p in enumerate(shuffled_patients)}
    
    # For each row in df_splits_input, store the integer index of its patient
    # (the one in 'shuffled_patients').
    df_splits_input["group_idx"] = df_splits_input["patient_id"].map(patient_to_idx)
    
    # We'll "simulate" data X = range(n_patients), y = 0 or something,
    # but the groups are indeed the group_idx.
    # Actually, we need an array where each sample is a single row from df,
    # but GroupKFold is typically at the row level. If we do so, we might end
    # up splitting the same patient across folds if we apply it directly.
    # 
    # Instead, let's do it at the PATIENT level: We'll pass X = range(n_patients)
    # and groups = range(n_patients) so that each patient is a single "sample".
    # Then for each fold, we gather the patient IDs (train, test).
    # 
    # This approach means each "split" gives us sets of *patients* in train/test.
    # Then we map those to actual filenames. 
    
    X = np.arange(n_patients)
    groups = X  # each patient is its own group
    
    for train_patient_idxs, test_patient_idxs in gkf.split(X, y=None, groups=groups):
        # The actual patient IDs
        train_patients = shuffled_patients[train_patient_idxs]
        test_patients  = shuffled_patients[test_patient_idxs]

        # Filenames that belong to those train/test patients
        train_filenames = df_splits_input[
            df_splits_input["patient_id"].isin(train_patients)
        ]["filename"].tolist()

        test_filenames = df_splits_input[
            df_splits_input["patient_id"].isin(test_patients)
        ]["filename"].tolist()

        cv_splits.append({
            "train": train_filenames,
            "test":  test_filenames
        })
    
    # -------------------------------------------------
    # 3) Final train/test1/test2 split at patient level
    # -------------------------------------------------
    train_prop, test1_prop, test2_prop = train_val_test_split_proportions

    train_end = int(train_prop * n_patients)
    test1_end = int(test1_prop * n_patients) + train_end

    train_patients_final = shuffled_patients[:train_end]
    test1_patients_final = shuffled_patients[train_end:test1_end]
    test2_patients_final = shuffled_patients[test1_end:]

    # Gather corresponding filenames for each set
    df_train = df_splits_input[df_splits_input["patient_id"].isin(train_patients_final)]
    df_test1 = df_splits_input[df_splits_input["patient_id"].isin(test1_patients_final)]
    df_test2 = df_splits_input[df_splits_input["patient_id"].isin(test2_patients_final)]
    
    # Convert to lists of filenames
    training_filenames = df_train["filename"].tolist()
    test1_filenames = df_test1["filename"].tolist()
    test2_filenames = df_test2["filename"].tolist()

    # -------------------------------------------------
    # 4) Build and return the final dictionary
    # -------------------------------------------------
    result = {
        "cv": cv_splits,
        "training": training_filenames,
        "test1": test1_filenames,
        "test2": test2_filenames
    }
    return result


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# base_dir = Path("/home/cwatzenboeck/data/AutoPIX_cirdata/")
# base_dir = Path("/home/clemens/data/AutoPIX_cirdata/")
base_dir = Path("/home/clemens/data/AutoPIX_cirdata_local/")



dataHandler = ra_utils.data.data_handler.DataHandler_CR_autoscoRA(
                folder_H_images = base_dir / "projects__autoscora/autoscoRA_images/H_images_of_interest_2_renamed_mirrored_inverted_dicoms",
                folder_F_images = base_dir / "projects__autoscora/autoscoRA_images/F_images_of_interest_2_renamed_mirrored_inverted_dicoms",
                df_lm_labels_H = base_dir / "projects__autoscora/landmark_data/100_all_H_joints36/points.csv",
                df_lm_labels_F = base_dir / "projects__autoscora/landmark_data/100_all_F_joints27/points.csv",
                df_autoscoRA_labels_F = base_dir / "projects__autoscora/autoscoRA_data/autoscoRA_feet.csv",
                df_autoscoRA_labels_H = base_dir / "projects__autoscora/autoscoRA_data/autoscoRA_hands.csv",
                training_test_splits_json_H = base_dir / "projects__autoscora/landmark_data/splits/splits_H_TD_25-03-05.json",
                training_test_splits_json_F = base_dir / "projects__autoscora/landmark_data/splits/splits_F_TD_25-03-05.json",
            )





In [ ]:
#  Make splits file 
df_splits_input = dataHandler.df_lm_labels_F[["filename"]].copy()
df_splits_input["patient_id"] = df_splits_input["filename"].apply(lambda x: x.split("_")[0])
 


# Suppose df_splits_input = dataHandler.df_lm_labels_H[["filename"]].copy()
split_dict = generate_split_dictionary(df_splits_input, 
                                       cw_splits=3,
                                       train_val_test_split_proportions=(0.6,0.2,0.2),
                                       random_seed=42)

import json
dst = "/home/clemens/data/AutoPIX_cirdata_local/projects__autoscora/landmark_data/splits/splits_F_25-03-05.json"
with open(dst, 'w') as fp:
    json.dump(split_dict, fp, sort_keys=True, indent=4)
    
    
#------------------------------------------


df_splits_input = dataHandler.df_lm_labels_H[["filename"]].copy()
df_splits_input["patient_id"] = df_splits_input["filename"].apply(lambda x: x.split("_")[0])

# Suppose df_splits_input = dataHandler.df_lm_labels_H[["filename"]].copy()
split_dict = generate_split_dictionary(df_splits_input, 
                                       cw_splits=3,
                                       train_val_test_split_proportions=(0.6,0.2,0.2),
                                       random_seed=42)


dst = "/home/clemens/data/AutoPIX_cirdata_local/projects__autoscora/landmark_data/splits/splits_H_25-03-05.json"
with open(dst, 'w') as fp:
    json.dump(split_dict, fp, sort_keys=True, indent=4)
    
    

In [ ]:
df = dataHandler.df_images_and_landmarks_H
# df["image"]
# df_dcm_infos = get_dicom_info(df["image"])



In [ ]:
### Data check
# Is are the landmarks positioned as I expect prior to resizing? 
# Check a single image: 
i=0
image_path = str(df["image"].iloc[i])
lm_sample = ra_utils.data.data_utils.extract_landmarks_from_df(df, image_idx=i)

image = LoadImage(ensure_channel_first=True)(image_path)
image = Transpose((0,2,1))(image)
ra_utils.visualization.plot_landmarks.plot_landmarks(image[0,...].numpy(), 
                                                     landmarks=lm_sample, 
                                                     figsize=(4,3));


In [ ]:
from landmarker.transforms.images import UseOnlyFirstChannel
fn_keys = ('image',)
spatial_transformd = [RandAffined(fn_keys, prob=1,
                        rotate_range=(-np.pi/12, np.pi/12),
                        translate_range=(-10, 10),
                        scale_range=(-0.1, 0.1),
                        shear_range=(-0.1, 0.1)
                        )]

train_transformd = Compose([
                            UseOnlyFirstChannel(('image', )),
                            Transposed(keys=["image"], indices=(0, 2, 1)), # CW added
                            RandGaussianNoised(('image', ), prob=0.2, mean=0, std=0.1),  # Add gaussian noise
                            RandScaleIntensityd(('image', ), factors=0.25, prob=0.2),  # Add random intensity scaling
                            RandAdjustContrastd(('image', ), prob=0.2, gamma=(0.5,4.5)),  # Randomly adjust contrast
                            RandHistogramShiftd(('image', ), prob=0.2),  # Randomly shift histogram
                            ScaleIntensityd(('image', )),  # Scale intensity
                        ] + spatial_transformd)

inference_transformd = Compose([
    UseOnlyFirstChannel(('image', )),
    Transposed(keys=["image"], indices=(0, 2, 1)),  # CW added
    ScaleIntensityd(('image', )),
])




In [ ]:

dim_image = (512,512)

(
    image_paths_train,
    image_paths_test1,
    image_paths_test2,
    landmarks_train,
    landmarks_test1,
    landmarks_test2
) = dataHandler.get_landmarks_dataset_H()

ds_train, ds_test1, ds_test2 = ra_utils.data.dataloader_CR_landmarks.get_landmark_datasets(
    image_paths_train = image_paths_train,
    image_paths_test1 = image_paths_test1,
    image_paths_test2 = image_paths_test2,
    landmarks_train = landmarks_train,
    landmarks_test1 = landmarks_test1,
    landmarks_test2 = landmarks_test2,
    train_transform=train_transformd,
    inference_transform=inference_transformd,
    dim_img=(512,512)
    )

N_landmarks = landmarks_train.shape[1]


In [ ]:
ds_train[0]["image"].shape

In [ ]:

# df = dataHandler.df_images_and_landmarks_H
# # image_paths: 

# image_paths = df["image"].apply(str).to_list()
# landmarks_array = [ra_utils.data.data_utils.extract_landmarks_from_df(df, image_idx=image_idx) 
#                    for image_idx, row in df.iterrows()]
# landmarks_array = np.array(landmarks_array, dtype=np.uint16)
# landmarks_array = np.flip(landmarks_array, axis=-1)

# landmarks_array.shape
# image_paths
# # landmarks_array = np.ones_like(landmarks_array)*128

# # N_data = landmarks_array.shape[0]
# # N_landmarks = landmarks_array.shape[1]


# landmarks_array.shape

In [ ]:


# #dim_image = (1687, 1283)
# dim_image = (512,512)

In [ ]:



# # Initialize dataset
# N_landmarks = landmarks_array.shape[1]

# # This works, but only sort of ... 
# # dataset = LandmarkDataset(
# #     imgs=image_paths,          # List of paths to your images
# #     landmarks=landmarks_array[..., ::-1], # NumPy array of shape (N, C, D)
# #     # landmarks=landmarks_array, # NumPy array of shape (N, C, D)
# #                              # N = number of samples
# #                              # C = number of landmark classes
# #                              # D = spatial dimensions (2 or 3)
# #     spatial_dims=2,          # 2 for 2D images, 3 for 3D
# #     transform=inference_transformd,    # MONAI transforms for preprocessing
# #     dim_img=dim_image,     # Target image dimensions
# # )

# #landmarks_array = np.concatenate(landmarks_array, axis=0).reshape((-1, N_landmarks, 2))
# landmarks_array_flipped = np.flip(landmarks_array, axis=-1)


# # This is what I want to get working ... 
# dataset = LandmarkDataset(
#     imgs=image_paths,          # List of paths to your images
#     landmarks=landmarks_array_flipped, # NumPy array of shape (N, C, D)
#                              # N = number of samples
#                              # C = number of landmark classes
#                              # D = spatial dimensions (2 or 3)
#     spatial_dims=2,          # 2 for 2D images, 3 for 3D
#     transform=train_transformd,
#     dim_img=dim_image,     # Target image dimensions
# )


# # ds_train, ds_test1, ds_test2 = torch.utils.data.random_split(dataset, [0.6, 0.2, 0.2], generator=torch.Generator().manual_seed(42))
# ds_train = dataset




In [ ]:
X = ds_train[0]

X.keys()

In [ ]:
X["landmark"].shape, X["image"].numpy()[0,...].shape

In [ ]:

image=X["image"].numpy()[0,...]
landmarks = X["landmark"].numpy()

fig, ax = plt.subplots(figsize=(4,3))
ax.imshow(image, cmap='gray')
ax.scatter(landmarks[:, 1], landmarks[:, 0], 
            s=20, color='red', marker='x')
ax.axis("off")
plt.show()



In [ ]:
from landmarker.heatmap.generator import GaussianHeatmapGenerator

heatmap_generator = GaussianHeatmapGenerator(
    nb_landmarks=N_landmarks,
    sigmas=3,
    gamma=400,
    heatmap_size=dim_image,
    learnable=True, # If True, the heatmap generator will be trainable
)



In [ ]:
ds = ds_train
ds_idx = 0
img = ds.image_loader(ds.img_paths[ds_idx])

#img = img.detach().numpy().astype(np.uint8)
plt.imshow(img.numpy(force=True)[0, ...])#.astype(np.uint8))

In [ ]:


# Plot the first 3 images from the training set
inspection_plot(ds_train, range(3,5), heatmap_generator=heatmap_generator)



In [ ]:
# Plot the first 3 images from dataset without transforms
heatmap_generator.device = "cpu" # because dataset tensors are still on cpu
inspection_plot(ds_test1, range(3), heatmap_generator=heatmap_generator)
heatmap_generator.device = device # set the desired device back